# 🤖 Crypto ML Predictor — MAX POWER Training

Training model dengan GPU di Google Colab.
Model: LightGBM + XGBoost + LSTM + Voting Ensemble

**Cara pakai:**
1. Run semua cell dari atas ke bawah
2. Model otomatis ter-save
3. Download model → upload ke Termux

In [ ]:
# CELL 1: SETUP & INSTALL
!pip install -q yfinance lightgbm xgboost tensorflow scikit-learn pandas numpy requests joblib

import pandas as pd
import numpy as np
import requests
import joblib
import json
import os
from datetime import datetime
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print('✅ Setup complete!')

In [ ]:
# CELL 2: FETCH DATA
import yfinance as yf

print('📡 Fetching 5 years BTC data...')
btc = yf.download('BTC-USD', period='5y', interval='1d', progress=False)
df = btc[['Close', 'Volume']].copy()
df.columns = ['price', 'volume']
df.index = df.index.tz_localize(None) if df.index.tz else df.index
df['volume'] = df['volume'].fillna(0)
print(f'BTC: {len(df)} rows ({df.index[0].date()} → {df.index[-1].date()})')

# Fear & Greed
try:
    fng = requests.get('https://api.alternative.me/fng/?limit=1825', timeout=10).json()['data']
    fng_df = pd.DataFrame(fng)
    fng_df['date'] = pd.to_datetime(fng_df['timestamp'], unit='s').dt.normalize()
    fng_df['fng'] = fng_df['value'].astype(float)
    fng_df = fng_df.set_index('date').sort_index()
    fng_df = fng_df[~fng_df.index.duplicated(keep='last')].resample('D').ffill()
    df = df.join(fng_df[['fng']], how='left')
    print('✅ F&G loaded')
except:
    df['fng'] = 50
    print('⚠️ F&G failed, using 50')

# Macro
for ticker, name in [('DX-Y.NYB','dxy'), ('^GSPC','sp500'), ('GC=F','gold'), ('^VIX','vix')]:
    try:
        d = yf.download(ticker, period='5y', interval='1d', progress=False)['Close']
        d.index = d.index.tz_localize(None) if d.index.tz else d.index
        df = df.join(d.rename(name), how='left')
    except:
        pass

df = df.ffill().bfill()
print(f'Final: {len(df)} rows')

In [ ]:
# CELL 3: FEATURE ENGINEERING (200+ features)
print('🔧 Computing 200+ features...')

for d in [1,2,3,5,7,10,14,21,30,60,90,180,365]:
    df[f'ret_{d}d'] = df['price'].pct_change(d)
df['log_ret_1d'] = np.log(df['price'] / df['price'].shift(1))

for p in [3,5,7,10,14,20,25,30,50,75,100,150,200,250,365]:
    df[f'sma_{p}'] = df['price'].rolling(p).mean()
    df[f'price_vs_sma_{p}'] = (df['price'] - df[f'sma_{p}']) / df[f'sma_{p}']
    df[f'sma_{p}_slope'] = df[f'sma_{p}'].pct_change(5)

for p in [5,8,12,13,21,26,34,55,89,144,200]:
    df[f'ema_{p}'] = df['price'].ewm(span=p, adjust=False).mean()
    df[f'price_vs_ema_{p}'] = (df['price'] - df[f'ema_{p}']) / df[f'ema_{p}']

for fast,slow,signal in [(12,26,9),(8,21,5),(5,35,5)]:
    ema_f = df['price'].ewm(span=fast, adjust=False).mean()
    ema_s = df['price'].ewm(span=slow, adjust=False).mean()
    df[f'macd_{fast}_{slow}'] = ema_f - ema_s
    df[f'macd_{fast}_{slow}_signal'] = df[f'macd_{fast}_{slow}'].ewm(span=signal, adjust=False).mean()
    df[f'macd_{fast}_{slow}_hist'] = df[f'macd_{fast}_{slow}'] - df[f'macd_{fast}_{slow}_signal']

def calc_rsi(series, period):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

for p in [5,7,9,14,21,28]:
    df[f'rsi_{p}'] = calc_rsi(df['price'], p)
df['rsi_14_slope'] = df['rsi_14'].diff(5)
df['price_slope_5d'] = df['price'].pct_change(5)
df['rsi_divergence'] = (np.sign(df['price_slope_5d']) != np.sign(df['rsi_14_slope'])).astype(int)

for p in [10,20,30]:
    for mult in [1.5, 2.0, 2.5]:
        mid = df['price'].rolling(p).mean()
        std = df['price'].rolling(p).std()
        df[f'bb_{p}_{mult}_upper'] = mid + mult * std
        df[f'bb_{p}_{mult}_lower'] = mid - mult * std
        df[f'bb_{p}_{mult}_width'] = (mult * 2 * std) / mid
        df[f'bb_{p}_{mult}_pos'] = (df['price'] - df[f'bb_{p}_{mult}_lower']) / (df[f'bb_{p}_{mult}_upper'] - df[f'bb_{p}_{mult}_lower'])

for p in [7,14,21]:
    hl = df['price'].rolling(2).max() - df['price'].rolling(2).min()
    df[f'atr_{p}'] = hl.rolling(p).mean()
    df[f'atr_{p}_pct'] = df[f'atr_{p}'] / df['price']

for p in [3,5,7,14,21,30,60]:
    df[f'vol_sma_{p}'] = df['volume'].rolling(p).mean()
    df[f'vol_ratio_{p}'] = df['volume'] / df[f'vol_sma_{p}']
df['vol_change_1d'] = df['volume'].pct_change(1)
df['vol_change_7d'] = df['volume'].pct_change(7)
df['obv'] = (np.sign(df['price'].diff()) * df['volume']).cumsum()
df['obv_sma_20'] = df['obv'].rolling(20).mean()
df['obv_vs_sma'] = df['obv'] / df['obv_sma_20']

for p in [5,7,10,14,21,30,60]:
    df[f'volatility_{p}d'] = df['log_ret_1d'].rolling(p).std()
df['vol_regime'] = df['volatility_7d'] / df['volatility_30d']

for d in [1,3,5,7,10,14,21,30,60,90]:
    df[f'momentum_{d}d'] = df['price'] / df['price'].shift(d) - 1
for d in [10,20,30]:
    df[f'roc_{d}'] = (df['price'] - df['price'].shift(d)) / df['price'].shift(d) * 100

for p in [14,21]:
    rsi = df[f'rsi_{p}']
    rsi_min = rsi.rolling(p).min()
    rsi_max = rsi.rolling(p).max()
    df[f'stoch_rsi_{p}'] = (rsi - rsi_min) / (rsi_max - rsi_min).replace(0, np.nan)

for p in [14,21]:
    high = df['price'].rolling(p).max()
    low = df['price'].rolling(p).min()
    df[f'williams_r_{p}'] = (high - df['price']) / (high - low).replace(0, np.nan) * -100

for p in [14,20]:
    tp = df['price']
    tp_sma = tp.rolling(p).mean()
    tp_mad = tp.rolling(p).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    df[f'cci_{p}'] = (tp - tp_sma) / (0.015 * tp_mad)

for window in [20,50,100,200]:
    df[f'support_{window}'] = df['price'].rolling(window).min()
    df[f'resistance_{window}'] = df['price'].rolling(window).max()
    df[f'price_vs_support_{window}'] = (df['price'] - df[f'support_{window}']) / df[f'support_{window}']
    df[f'price_vs_resistance_{window}'] = (df['price'] - df[f'resistance_{window}']) / df[f'resistance_{window}']

df['trend_7d'] = np.where(df['price'] > df['sma_7'], 1, -1)
df['trend_20d'] = np.where(df['price'] > df['sma_20'], 1, -1)
df['trend_50d'] = np.where(df['price'] > df['sma_50'], 1, -1)
df['trend_200d'] = np.where(df['price'] > df['sma_200'], 1, -1)
df['trend_alignment'] = df['trend_7d'] + df['trend_20d'] + df['trend_50d'] + df['trend_200d']

df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
df['is_month_start'] = df.index.is_month_start.astype(int)
df['is_month_end'] = df.index.is_month_end.astype(int)
df['day_of_month'] = df.index.day

for col in ['dxy','sp500','gold','vix']:
    if col in df.columns:
        for d in [1,3,5,7,14,30]:
            df[f'{col}_ret_{d}d'] = df[col].pct_change(d)

if 'fng' in df.columns:
    df['fng_sma_7'] = df['fng'].rolling(7).mean()
    df['fng_sma_30'] = df['fng'].rolling(30).mean()
    df['fng_change'] = df['fng'].diff(7)
    df['fng_extreme_fear'] = (df['fng'] < 20).astype(int)
    df['fng_extreme_greed'] = (df['fng'] > 80).astype(int)

print(f'✅ Features: {len(df.columns)} columns')

In [ ]:
# CELL 4: TARGET AND FEATURE SELECTION
df['target_ret_7d'] = df['price'].shift(-7) / df['price'] - 1
df['target_dir_7d'] = (df['target_ret_7d'] > 0).astype(int)

exclude = ['price', 'volume',
           'sma_3','sma_5','sma_7','sma_10','sma_14','sma_20','sma_25','sma_30','sma_50','sma_75','sma_100','sma_150','sma_200','sma_250','sma_365',
           'ema_5','ema_8','ema_12','ema_13','ema_21','ema_26','ema_34','ema_55','ema_89','ema_144','ema_200',
           'bb_10_1.5_upper','bb_10_1.5_lower','bb_10_2.0_upper','bb_10_2.0_lower','bb_10_2.5_upper','bb_10_2.5_lower',
           'bb_20_1.5_upper','bb_20_1.5_lower','bb_20_2.0_upper','bb_20_2.0_lower','bb_20_2.5_upper','bb_20_2.5_lower',
           'bb_30_1.5_upper','bb_30_1.5_lower','bb_30_2.0_upper','bb_30_2.0_lower','bb_30_2.5_upper','bb_30_2.5_lower',
           'vol_sma_3','vol_sma_5','vol_sma_7','vol_sma_14','vol_sma_21','vol_sma_30','vol_sma_60',
           'atr_7','atr_14','atr_21','obv','obv_sma_20',
           'dxy','sp500','gold','vix',
           'support_20','support_50','support_100','support_200',
           'resistance_20','resistance_50','resistance_100','resistance_200',
           'target_ret_7d','target_dir_7d']

all_features = [c for c in df.columns if c not in exclude]
df_clean = df.dropna()
X_all = df_clean[all_features]
y = df_clean['target_dir_7d']

print('🔍 Selecting top 60 features...')
constant_cols = [c for c in X_all.columns if X_all[c].nunique() <= 1]
X_all = X_all.drop(columns=constant_cols, errors='ignore')

mi_scores = mutual_info_classif(X_all.fillna(0), y, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({'feature': X_all.columns, 'mi': mi_scores}).sort_values('mi', ascending=False)
top_features = mi_df.head(60)['feature'].tolist()

print('Top 10 features:')
for _, row in mi_df.head(10).iterrows():
    print(f'  {row["feature"]}: {row["mi"]:.4f}')

X = df_clean[top_features]
print(f'\n✅ Training data: {len(X)} samples, {len(top_features)} features')

In [ ]:
# CELL 5: TRAIN ML MODELS
tscv = TimeSeriesSplit(n_splits=5)
scaler = StandardScaler()
X_s = pd.DataFrame(scaler.fit_transform(X), columns=top_features, index=X.index)

results = {}

# XGBoost
print('\n🔄 Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=700, max_depth=6, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0
)
accs = []
for train_idx, test_idx in tscv.split(X_s):
    xgb_model.fit(X_s.iloc[train_idx], y.iloc[train_idx])
    accs.append(accuracy_score(y.iloc[test_idx], xgb_model.predict(X_s.iloc[test_idx])))
results['XGB'] = (np.mean(accs), xgb_model)
print(f'  XGB: {np.mean(accs):.4f}')

# LightGBM
print('\n🔄 Training LightGBM...')
lgb_model = lgb.LGBMClassifier(
    n_estimators=700, max_depth=6, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1
)
accs = []
for train_idx, test_idx in tscv.split(X_s):
    lgb_model.fit(X_s.iloc[train_idx], y.iloc[train_idx])
    accs.append(accuracy_score(y.iloc[test_idx], lgb_model.predict(X_s.iloc[test_idx])))
results['LGBM'] = (np.mean(accs), lgb_model)
print(f'  LGBM: {np.mean(accs):.4f}')

# Random Forest
print('\n🔄 Training Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=500, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1
)
accs = []
for train_idx, test_idx in tscv.split(X_s):
    rf_model.fit(X_s.iloc[train_idx], y.iloc[train_idx])
    accs.append(accuracy_score(y.iloc[test_idx], rf_model.predict(X_s.iloc[test_idx])))
results['RF'] = (np.mean(accs), rf_model)
print(f'  RF: {np.mean(accs):.4f}')

# Gradient Boosting
print('\n🔄 Training Gradient Boosting...')
gb_model = GradientBoostingClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, min_samples_split=10, random_state=42
)
accs = []
for train_idx, test_idx in tscv.split(X_s):
    gb_model.fit(X_s.iloc[train_idx], y.iloc[train_idx])
    accs.append(accuracy_score(y.iloc[test_idx], gb_model.predict(X_s.iloc[test_idx])))
results['GB'] = (np.mean(accs), gb_model)
print(f'  GB: {np.mean(accs):.4f}')

# Voting Ensemble
print('\n🔄 Training Voting Ensemble...')
voting = VotingClassifier(
    estimators=[
        ('xgb', xgb.XGBClassifier(n_estimators=700, max_depth=6, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0)),
        ('lgb', lgb.LGBMClassifier(n_estimators=700, max_depth=6, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)),
        ('rf', RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_split=10, min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)),
    ],
    voting='soft',
    weights=[1.2, 1.2, 1.0],
    n_jobs=-1
)
accs = []
for train_idx, test_idx in tscv.split(X_s):
    voting.fit(X_s.iloc[train_idx], y.iloc[train_idx])
    accs.append(accuracy_score(y.iloc[test_idx], voting.predict(X_s.iloc[test_idx])))
results['Voting'] = (np.mean(accs), voting)
print(f'  Voting: {np.mean(accs):.4f}')

# Results
print('\n' + '='*50)
print('📊 MODEL RESULTS:')
print('='*50)
best_name = max(results, key=lambda k: results[k][0])
for name, (acc, _) in sorted(results.items(), key=lambda x: -x[1][0]):
    marker = '🏆' if name == best_name else '  '
    print(f'{marker} {name}: {acc:.4f}')
print(f'\n✅ Best: {best_name} ({results[best_name][0]:.4f})')

In [ ]:
# CELL 6: TRAIN LSTM (GPU)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print(f'GPU: {tf.config.list_physical_devices("GPU")}')

SEQ_LEN = 30

def create_sequences(X_data, y_data, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X_data)):
        Xs.append(X_data[i-seq_len:i])
        ys.append(y_data[i])
    return np.array(Xs), np.array(ys)

X_scaled = scaler.fit_transform(X)
y_arr = y.values
X_seq, y_seq = create_sequences(X_scaled, y_arr, SEQ_LEN)
print(f'Sequences: {X_seq.shape}')

split_idx = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

model_lstm = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, X_seq.shape[2])),
    Dropout(0.3),
    BatchNormalization(),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    BatchNormalization(),
    LSTM(32, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

print('\n🔄 Training LSTM...')
history = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[
        EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
        ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
    ],
    verbose=1
)

lstm_preds = (model_lstm.predict(X_test) > 0.5).astype(int).flatten()
lstm_acc = accuracy_score(y_test, lstm_preds)
print(f'\n✅ LSTM Accuracy: {lstm_acc:.4f}')

results['LSTM'] = (lstm_acc, model_lstm)
best_name = max(results, key=lambda k: results[k][0])
print(f'\n🏆 Overall Best: {best_name} ({results[best_name][0]:.4f})')

In [ ]:
# CELL 7: PREDICT AND SAVE
best_acc, best_model = results[best_name]

if best_name == 'LSTM':
    latest_seq = X_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, -1)
    prob_up = model_lstm.predict(latest_seq)[0][0]
    pred = 1 if prob_up > 0.5 else 0
    proba = [1-prob_up, prob_up]
else:
    latest_s = scaler.transform(X.iloc[-1:])
    pred = best_model.predict(latest_s)[0]
    proba = best_model.predict_proba(latest_s)[0]

btc_price = float(df['price'].iloc[-1])
direction = 'NAIK' if pred == 1 else 'TURUN'
conf = max(proba) * 100

print('='*50)
print('📊 PREDICTION')
print('='*50)
print(f'💰 BTC: ${btc_price:,.2f}')
print(f'📈 Direction: {direction}')
print(f'🎯 Confidence: {conf:.1f}%')
print(f'📊 Prob NAIK: {proba[1]*100:.1f}% | TURUN: {proba[0]*100:.1f}%')
print(f'🤖 Model: {best_name} ({best_acc*100:.1f}% acc)')

os.makedirs('models', exist_ok=True)

if best_name != 'LSTM':
    joblib.dump(best_model, 'models/crypto_model.pkl')
    joblib.dump(scaler, 'models/crypto_scaler.pkl')
    import pickle
    with open('models/feature_cols.pkl', 'wb') as f:
        pickle.dump(top_features, f)

model_lstm.save('models/lstm_model.h5')

result = {
    'btc_price': btc_price,
    'direction': direction,
    'confidence': float(conf),
    'prob_up': float(proba[1]),
    'prob_down': float(proba[0]),
    'model_name': best_name,
    'accuracy': float(best_acc),
    'timestamp': datetime.now().isoformat()
}
with open('models/prediction.json', 'w') as f:
    json.dump(result, f, indent=2)

print('\n✅ Models saved to /models/')

In [ ]:
# CELL 8: DOWNLOAD
from google.colab import files
import zipfile

with zipfile.ZipFile('crypto_models.zip', 'w') as zf:
    for f in os.listdir('models'):
        zf.write(f'models/{f}', f)

print('📦 Downloading models...')
files.download('crypto_models.zip')
print('✅ Done! Upload zip ke Termux lo.')